# TL5 — Regresión logística

## Índice

1. [Introducción](#1-introducción)
2. [Regresión logística](#2-regresión-logística)
3. [Ingeniería de características](#3-ingeniería-de-características)
4. [Aplicación a MNIST](#4-aplicación-a-mnist)
5. [Ejercicio](#5-ejercicio)

> **Cita:** Juan Ciscar, Alfons. *"Material docente de la asignatura Aprendizaje Automático (APA)"*. Grado de Inteligencia Artificial (GIAR), Escuela Técnica Superior de Ingeniería Informática (ETSINF), Dep. de Sistemas Informáticos y Computación (DSIC), Universitat Politècnica de València (UPV), primer semestre del curso 2026/27.

---
## 1. Introducción

**Clasificador discriminativo:** modela *posteriors* directamente; no requiere *priors* ni densidades condicionales.

$$
p(y = c \mid \boldsymbol{x}; \boldsymbol{\theta}) = \cdots
$$

**Ingeniería de características:** la entrada $\boldsymbol{x}$ se sustituye por alguna transformación $\boldsymbol{\phi}(\boldsymbol{x})$.

**Objetivo:** aplicar regresión logística con ingeniería de características.

---
## 2. Regresión logística

**Regresión logística con ingeniería de características:** $\mathbf{W} \in \mathbb{R}^{D \times C}$ es la matriz de pesos

$$
p(y \mid \boldsymbol{x}, \mathbf{W}) = \operatorname{Cat}(y \mid \boldsymbol{\mu})
\quad \text{con} \quad \boldsymbol{\mu} = \mathcal{S}(\boldsymbol{a})
\quad \text{y} \quad \boldsymbol{a} = \mathbf{W}^t \boldsymbol{x}
$$

donde $\mathbf{W}$ se ajusta minimizando la NLL mediante descenso por gradiente.

**Regresión logística en scikit-learn:** `LogisticRegression`

| Parámetro | Descripción |
|-----------|-------------|
| `tol=1e-4` | tolerancia del criterio de parada |
| `C=1.0` | parámetro de des-regularización (cuanto mayor, menor regularización) |
| `fit_intercept=True` | añade un parámetro de sesgo a $\boldsymbol{a}$ |
| `solver='lbfgs'` | técnica de optimización (descenso por gradiente) |
| `max_iter=100` | máximo número de iteraciones |

---
## 3. Ingeniería de características

**Objetivo:** encontrar alguna transformación de los datos, $\boldsymbol{\phi}(\boldsymbol{x})$, que linealice un problema de clasificación con clases (datos) no linealmente separables.

**Características polinómicas:** $\boldsymbol{x}$ se sustituye por una expansión polinómica hasta un grado dado.

**Características polinómicas en scikit-learn:** `PolynomialFeatures`

| Parámetro | Descripción |
|-----------|-------------|
| `degree=2` | grado máximo de las características generadas, o par "(mínimo, máximo)" |
| `interaction_only=False` | características interactivas (productos de características distintas) solo |
| `include_bias=True` | notación homogénea |

**Ejemplo:** $\boldsymbol{x} = (x_1, x_2)^t \;\Rightarrow\; (1, x_1, x_2, x_1^2, x_1 x_2, x_2^2)^t$, $\;(1, x_1, x_2, x_1 x_2)^t\;$ y $\;(1, x_1, x_2, x_1^2, x_2^2)^t$

### 3.1 Expansión polinómica completa

Salida esperada: `[[2. 3.]] => [[1. 2. 3. 4. 6. 9.]]`

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

X = np.arange(2, 4).astype(np.float64).reshape(-1, 2)
print(X, '=>', PolynomialFeatures(2).fit_transform(X))

### 3.2 Solo interacciones

Salida esperada: `[[2. 3.]] => [[1. 2. 3. 6.]]`

In [ ]:
print(X, '=>', PolynomialFeatures(2, interaction_only=True).fit_transform(X))

### 3.3 Solo cuadrados (sin interacciones)

Salida esperada: `[[2. 3.]] => [[1. 2. 3. 4. 9.]]`

In [ ]:
print(X, '=>', np.c_[(np.ones((X.shape[0], 1)), X, np.square(X))])

### 3.4 Ejemplo: tarea sintética con clases no linealmente separables

Sin y con ingeniería de características.

Salida esperada: `Precisión: 50.4%, con características polinómicas 99.2%`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=1000, factor=0.2, noise=0.2, random_state=23)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=23)

clf = LogisticRegression(random_state=23).fit(X_train, y_train)
acc = clf.score(X_test, y_test)

clfp = make_pipeline(StandardScaler(), PolynomialFeatures(), LogisticRegression(random_state=23))
clfp.fit(X_train, y_train)
accp = clfp.score(X_test, y_test)

print(f'Precisión: {acc:.1%}, con características polinómicas {accp:.1%}')

Fronteras de decisión: sin (izquierda) y con (derecha) características polinómicas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

R = np.linspace(-1.75, 1.75, 200)
X1, X2 = np.meshgrid(R, R)
X = np.c_[np.ravel(X1), np.ravel(X2)]

fig, (ax, axp) = plt.subplots(1, 2, figsize=(5, 2.5), constrained_layout=True)
ax.contourf(X1, X2, clf.predict(X).reshape(X1.shape), 6, cmap='Blues_r')
ax.scatter(*X_test.T, c=y_test, cmap='Blues_r', edgecolors='C1')
axp.contourf(X1, X2, clfp.predict(X).reshape(X1.shape), 6, cmap='Blues_r')
axp.scatter(*X_test.T, c=y_test, cmap='Blues_r', edgecolors='C1')
plt.show()

---
## 4. Aplicación a MNIST

### 4.1 Lectura y reducción de la dimensión ($K = 400$)

Solo el *train*; el *test* se carga después para ahorrar memoria.

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)

K = 400
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)

### 4.2 Entrenamiento con características cuadráticas

Salida esperada:

```text
(60000, 80601) (60000,)
Tiempo (hh:mm:ss): 00:01:17
```

In [ ]:
import time
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LogisticRegression

start = time.time()

poly = PolynomialFeatures(2)
X_train = poly.fit_transform(X_train)
print(X_train.shape, y_train.shape)

clf = LogisticRegression(C=0.1, max_iter=200).fit(X_train, y_train)
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

del X_train, y_train

### 4.3 Evaluación en test

Salida esperada:

```text
(10000, 80601) (10000,)
Precisión: 98.3%
```

In [ ]:
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
X_test = pca.transform(X_test)
X_test = poly.transform(X_test)
print(X_test.shape, y_test.shape)

acc = clf.score(X_test, y_test)
print(f'Precisión: {acc:.1%}')

---
## 5. Ejercicio

**Ejercicio:** aplica LR con características cuadráticas (QLR) a Fashion-MNIST y CIFAR-10 con $K = 400$.

**Precisión (%) en test:** QLR obtiene los mejores resultados en MNIST y Fashion-MNIST; próximo al mejor en CIFAR-10.

|         | MNIST<br>$K=400$ | Fashion<br>$K=600$ | Fashion<br>$K=400$ | CIFAR-10<br>$K=1000$ | CIFAR-10<br>$K=400$ |
|---------|:----:|:----:|:----:|:----:|:----:|
| **LR**  | 92.6 | 84.4 | −    | 41.1 | −    |
| **LDA** | 87.3 | 81.6 | −    | 40.0 | −    |
| **QDA** | 95.8 | 76.6 | −    | 53.0 | −    |
| **QLR** | 98.3 | −    | 89.4 | −    | 52.6 |

### 5.1 Fashion-MNIST ($K = 400$)

#### 5.1.1 Lectura y reducción de la dimensión (solo *train*)

In [ ]:
# Tu código aquí

#### 5.1.2 Entrenamiento con características cuadráticas

Salida esperada:

```text
(60000, 80601) (60000,)
Tiempo (hh:mm:ss): 00:05:09
```

In [ ]:
# Tu código aquí

#### 5.1.3 Evaluación en test

Salida esperada:

```text
(10000, 80601) (10000,)
Precisión: 89.4%
```

In [ ]:
# Tu código aquí

### 5.2 CIFAR-10 ($K = 400$)

#### 5.2.1 Lectura y reducción de la dimensión (solo *train*)

In [ ]:
# Tu código aquí

#### 5.2.2 Entrenamiento con características cuadráticas

Salida esperada:

```text
(50000, 80601) (50000,)
Tiempo (hh:mm:ss): 00:03:42
```

In [ ]:
# Tu código aquí

#### 5.2.3 Evaluación en test

Salida esperada:

```text
(10000, 80601) (10000,)
Precisión: 52.6%
```

In [ ]:
# Tu código aquí